# Noise models

反復測定から観測分散を推定する `ReplicateNoiseSingleTaskGP` を使い、latent function の不確かさと測定ノイズを分離します。


## 反復測定データ


In [ ]:
import torch
from robotorchan.models import ReplicateNoiseSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
base_X = torch.tensor([[0.0], [0.5], [1.0]])
X = base_X.repeat_interleave(3, dim=0)
Y = torch.sin(2 * torch.pi * X) + 0.05 * torch.randn_like(X)


## 反復を集約してGPを構築


In [ ]:
model = ReplicateNoiseSingleTaskGP.from_replicates(X, Y)
model.fit()
print(model.raw_train_X)
print(model.raw_train_Yvar)
print(model.replicate_counts)


## Posterior


In [ ]:
test_X = torch.linspace(0, 1, 5).unsqueeze(-1)
posterior = model.posterior(test_X)
print(posterior.mean.squeeze(-1))


## Replicate-aware candidate proposal

反復観測から推定したノイズ分散を持つモデルで、予測平均最大の候補を選択します。


In [ ]:
candidate_grid = torch.linspace(0, 1, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
candidate_Y = torch.sin(2 * torch.pi * candidate)
assert candidate.shape == (1, 1)
print("candidate:", candidate, "evaluated objective:", candidate_Y)
